In [ ]:
log_dir_config = {
        'HistGen': 'histgen_training_*',
        'TITAN': 'TITAN_histgen_training_*',
        'UNI': 'uni1_training_*',
        'UNI2': 'uni2_training_*'
    }

In [3]:
# Seperate training and validation plots

import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob
import os


# Model color scheme (consistent with previous plots)
MODEL_COLORS = {
    'HistGen': '#7f8c8d',
    'UNI': '#3498db',
    'UNI2': '#8e44ad',
    'TITAN': '#e74c3c'
}


def parse_histgen_log_robust(log_file_path, debug=False):
    """
    More robust parser that handles various log formats
    """

    training_data = {
        'epochs': [],
        'train_losses': [],
        'val_BLEU_1': [],
        'val_BLEU_2': [],
        'val_BLEU_3': [],
        'val_BLEU_4': [],
        'val_METEOR': [],
        'val_ROUGE_L': [],
        'best_epochs': []
    }

    with open(log_file_path, 'r') as f:
        content = f.read()

    # Split by "Best results" to ignore final summary
    if "Best results (w.r.t BLEU_4)" in content:
        content = content.split("Best results (w.r.t BLEU_4)")[0]

    lines = content.split('\n')

    # Find all epoch blocks
    epoch_blocks = []
    current_block = []
    in_epoch = False

    for line in lines:
        # Detect start of epoch training
        if re.search(r'\[(\d+)/(\d+)\] Start to train in the training set', line):
            if current_block:
                epoch_blocks.append('\n'.join(current_block))
            current_block = [line]
            in_epoch = True
        elif in_epoch:
            current_block.append(line)
            # Detect end of epoch (saving checkpoint or next epoch start)
            if 'Saving checkpoint:' in line or 'Saving new best model:' in line:
                # Check if we have all metrics
                block_text = '\n'.join(current_block)
                if 'val_BLEU_4' in block_text:
                    in_epoch = False

    if current_block:
        epoch_blocks.append('\n'.join(current_block))

    if debug:
        print(f"  Found {len(epoch_blocks)} epoch blocks")

    # Parse each epoch block
    for block in epoch_blocks:
        # Extract epoch number
        epoch_match = re.search(r'\tepoch\s+:\s+(\d+)', block)
        if not epoch_match:
            continue

        epoch_num = int(epoch_match.group(1))

        # Extract metrics
        metrics = {}

        # Training loss
        train_loss_match = re.search(r'\ttrain_loss\s+:\s+([\d.]+)', block)
        if train_loss_match:
            metrics['train_loss'] = float(train_loss_match.group(1))

        # Validation metrics
        for metric_name in ['BLEU_1', 'BLEU_2', 'BLEU_3', 'BLEU_4', 'METEOR', 'ROUGE_L']:
            pattern = f'\tval_{metric_name}\s+:\s+([\d.]+)'
            match = re.search(pattern, block)
            if match:
                metrics[f'val_{metric_name}'] = float(match.group(1))

        # Only save if we have all required metrics
        required_metrics = ['train_loss', 'val_BLEU_1', 'val_BLEU_2', 'val_BLEU_3', 
                          'val_BLEU_4', 'val_METEOR', 'val_ROUGE_L']

        if all(m in metrics for m in required_metrics):
            training_data['epochs'].append(epoch_num)
            training_data['train_losses'].append(metrics['train_loss'])
            training_data['val_BLEU_1'].append(metrics['val_BLEU_1'])
            training_data['val_BLEU_2'].append(metrics['val_BLEU_2'])
            training_data['val_BLEU_3'].append(metrics['val_BLEU_3'])
            training_data['val_BLEU_4'].append(metrics['val_BLEU_4'])
            training_data['val_METEOR'].append(metrics['val_METEOR'])
            training_data['val_ROUGE_L'].append(metrics['val_ROUGE_L'])

        # Check for best model
        if 'Saving new best model:' in block:
            best_match = re.search(r'model_best_epoch(\d+)\.pth', block)
            if best_match:
                training_data['best_epochs'].append(int(best_match.group(1)))

    if debug and len(training_data['epochs']) > 0:
        print(f"  Successfully parsed epochs: {training_data['epochs'][:5]}... (first 5)")
        print(f"  Sample train_loss: {training_data['train_losses'][:3]}")
        print(f"  Sample val_BLEU_4: {training_data['val_BLEU_4'][:3]}")

    return training_data


def aggregate_training_data(log_files):
    """
    Parse multiple log files and aggregate statistics across seeds
    """
    all_data = []

    for log_file in log_files:
        print(f"  Parsing: {os.path.basename(log_file)}")
        try:
            data = parse_histgen_log_robust(log_file, debug=True)
            if len(data['epochs']) > 0:
                all_data.append(data)
                print(f"    ✓ Found {len(data['epochs'])} epochs")
            else:
                print(f"    ⚠ No valid epochs found")

                # Extra debugging for failed file
                with open(log_file, 'r') as f:
                    content = f.read()

                print(f"    Debug info:")
                print(f"      - File size: {len(content)} chars")
                print(f"      - Contains 'Start to train': {('Start to train' in content)}")
                print(f"      - Contains 'val_BLEU_4': {('val_BLEU_4' in content)}")
                print(f"      - Sample line with epoch:")
                for line in content.split('\n')[:200]:  # Check first 200 lines
                    if 'epoch' in line.lower() and ':' in line:
                        print(f"        {line.strip()}")
                        break

        except Exception as e:
            print(f"    ✗ Error: {e}")
            import traceback
            traceback.print_exc()

    if len(all_data) == 0:
        raise ValueError("No valid training data found in log files")

    # Find maximum epochs across all seeds
    max_epochs = max(len(d['epochs']) for d in all_data)

    # Aggregate data
    aggregated = {
        'epochs': list(range(1, max_epochs + 1)),
        'metrics': {}
    }

    metric_names = ['train_losses', 'val_BLEU_1', 'val_BLEU_2', 'val_BLEU_3', 
                   'val_BLEU_4', 'val_METEOR', 'val_ROUGE_L']

    for metric in metric_names:
        # Collect values for each epoch across seeds
        values_per_epoch = []
        for epoch_idx in range(max_epochs):
            epoch_values = []
            for seed_data in all_data:
                if epoch_idx < len(seed_data[metric]):
                    epoch_values.append(seed_data[metric][epoch_idx])
            values_per_epoch.append(epoch_values)

        # Calculate mean and std for each epoch
        means = [np.mean(vals) if len(vals) > 0 else np.nan for vals in values_per_epoch]
        stds = [np.std(vals) if len(vals) > 1 else 0 for vals in values_per_epoch]

        aggregated['metrics'][metric] = {
            'mean': means,
            'std': stds,
            'raw': values_per_epoch
        }

    # Aggregate best epochs
    all_best_epochs = []
    for seed_data in all_data:
        all_best_epochs.extend(seed_data['best_epochs'])
    aggregated['best_epochs'] = all_best_epochs

    return aggregated


def plot_aggregated_training_curves(models_data, output_prefix='training_curves'):
    """
    Create publication-quality training curves with aggregated data
    SAVES TRAINING AND VALIDATION PLOTS SEPARATELY
    """

    # Plot 1: Individual model plots (Training and Validation saved separately)
    for model_name, data in models_data.items():
        epochs = data['epochs']
        color = MODEL_COLORS[model_name]

        # ===== TRAINING LOSS PLOT (SEPARATE) =====
        fig_train = plt.figure(figsize=(8, 6))
        ax1 = fig_train.add_subplot(111)

        train_mean = data['metrics']['train_losses']['mean']
        train_std = data['metrics']['train_losses']['std']

        ax1.plot(epochs, train_mean, color=color, linewidth=2.5, label=f'{model_name} Mean')
        ax1.fill_between(epochs, 
                         np.array(train_mean) - np.array(train_std),
                         np.array(train_mean) + np.array(train_std),
                         alpha=0.2, color=color, label='±1 Std')

        ax1.set_xlabel('Epoch', fontsize=13, fontweight='bold')
        ax1.set_ylabel('Training Loss', fontsize=13, fontweight='bold')
        ax1.set_title(f'{model_name}: Training Loss Convergence', 
                    fontsize=14, fontweight='bold')
        ax1.set_yscale('log')

        # Better y-axis ticks for log scale
        ax1.set_yticks([0.05, 0.1, 0.2, 0.3, 0.5, 0.7])
        ax1.set_yticklabels(['0.05', '0.1', '0.2', '0.3', '0.5', '0.7'])
        ax1.set_ylim([0.04, 0.8])

        ax1.grid(True, which='major', alpha=0.5, linestyle='-', linewidth=0.8)
        ax1.grid(True, which='minor', alpha=0.2, linestyle='--', linewidth=0.5)
        ax1.tick_params(axis='both', which='major', labelsize=11)
        ax1.legend(fontsize=11)

        plt.tight_layout()
        plt.savefig(f'{output_prefix}_{model_name}_training_loss.pdf', dpi=300, bbox_inches='tight')
        plt.savefig(f'{output_prefix}_{model_name}_training_loss.png', dpi=300, bbox_inches='tight')
        print(f"  ✓ Saved training loss plot for {model_name}")
        plt.close(fig_train)

        # ===== VALIDATION BLEU-4 PLOT (SEPARATE) =====
        fig_val = plt.figure(figsize=(8, 6))
        ax2 = fig_val.add_subplot(111)

        val_mean = data['metrics']['val_BLEU_4']['mean']
        val_std = data['metrics']['val_BLEU_4']['std']

        ax2.plot(epochs, val_mean, color=color, linewidth=2.5, label=f'{model_name} Mean')
        ax2.fill_between(epochs,
                         np.array(val_mean) - np.array(val_std),
                         np.array(val_mean) + np.array(val_std),
                         alpha=0.2, color=color, label='±1 Std')

        # Mark best epochs
        if data['best_epochs']:
            from collections import Counter
            best_epoch_counts = Counter(data['best_epochs'])
            most_common_best = best_epoch_counts.most_common(3)

            for best_ep, count in most_common_best:
                if best_ep <= len(val_mean):
                    ax2.axvline(x=best_ep, color='red', linestyle='--', alpha=0.5, linewidth=1.5)
                    ax2.text(best_ep, ax2.get_ylim()[1]*0.95, f'E{best_ep}\n({count})', 
                            ha='center', fontsize=9, color='red')

        ax2.set_xlabel('Epoch', fontsize=13, fontweight='bold')
        ax2.set_ylabel('Validation BLEU-4', fontsize=13, fontweight='bold')
        ax2.set_title(f'{model_name}: Validation BLEU-4 Evolution', 
                     fontsize=14, fontweight='bold')
        ax2.grid(True, alpha=0.3, linestyle='--')
        ax2.legend(fontsize=11)
        ax2.set_ylim([0, 0.8])

        plt.tight_layout()
        plt.savefig(f'{output_prefix}_{model_name}_validation_bleu4.pdf', dpi=300, bbox_inches='tight')
        plt.savefig(f'{output_prefix}_{model_name}_validation_bleu4.png', dpi=300, bbox_inches='tight')
        print(f"  ✓ Saved validation BLEU-4 plot for {model_name}")
        plt.close(fig_val)

    # Plot 2: Combined comparison (unchanged)
    fig, ax = plt.subplots(figsize=(12, 7))

    for model_name, data in models_data.items():
        epochs = data['epochs']
        val_mean = data['metrics']['val_BLEU_4']['mean']
        val_std = data['metrics']['val_BLEU_4']['std']
        color = MODEL_COLORS[model_name]

        ax.plot(epochs, val_mean, color=color, linewidth=2.5, label=model_name, marker='o', markersize=4)
        ax.fill_between(epochs,
                        np.array(val_mean) - np.array(val_std),
                        np.array(val_mean) + np.array(val_std),
                        alpha=0.15, color=color)

    ax.set_xlabel('Epoch', fontsize=13, fontweight='bold')
    ax.set_ylabel('Validation BLEU-4', fontsize=13, fontweight='bold')
    ax.set_title('Validation BLEU-4 Evolution across all Models', 
                fontsize=14, fontweight='bold', pad=15)
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.legend(fontsize=12, loc='lower right')
    ax.set_ylim([0.35, 0.7])

    plt.tight_layout()
    plt.savefig(f'{output_prefix}_all_models_comparison.pdf', dpi=300, bbox_inches='tight')
    plt.savefig(f'{output_prefix}_all_models_comparison.png', dpi=300, bbox_inches='tight')
    print(f"  ✓ Saved combined comparison plot")
    plt.close()


def create_training_plots_from_logs(log_dir_config, output_prefix='training_curves'):
    """
    Main function with enhanced debugging
    """

    print("="*70)
    print("PARSING TRAINING LOGS")
    print("="*70)

    models_data = {}

    for model_name, log_spec in log_dir_config.items():
        print(f"\nProcessing {model_name}:")

        # Get list of log files
        if isinstance(log_spec, str):
            log_files = glob.glob(log_spec)
        else:
            log_files = log_spec

        if len(log_files) == 0:
            print(f"  ⚠ No log files found for {model_name}")
            continue

        print(f"  Found {len(log_files)} log files")

        # Aggregate data across seeds
        try:
            aggregated_data = aggregate_training_data(log_files)
            models_data[model_name] = aggregated_data
            print(f"  ✓ Aggregated {len(log_files)} seeds, {len(aggregated_data['epochs'])} epochs")
        except Exception as e:
            print(f"  ✗ Error aggregating data: {e}")
            import traceback
            traceback.print_exc()

    if len(models_data) == 0:
        raise ValueError("No valid model data found")

    print("\n" + "="*70)
    print("GENERATING PLOTS")
    print("="*70)

    plot_aggregated_training_curves(models_data, output_prefix)

    print("\n" + "="*70)
    print("COMPLETED")
    print("="*70)


# Example usage
if __name__ == "__main__":
    log_dir_config = {
        'HistGen': 'histgen_training_*',
        'TITAN': 'TITAN_histgen_training_*',
        'UNI': 'uni1_training_*',
        'UNI2': 'uni2_training_*'
    }

    create_training_plots_from_logs(log_dir_config, output_prefix='training_curves')


PARSING TRAINING LOGS

Processing HistGen:
  Found 5 log files
  Parsing: histgen_training_17.o1202346
  Found 40 epoch blocks
  Successfully parsed epochs: [1, 2, 3, 4, 5]... (first 5)
  Sample train_loss: [0.6485917021111374, 0.2985893484672145, 0.25281591293946637]
  Sample val_BLEU_4: [0.38479442944291037, 0.48356529842690343, 0.4844273691166674]
    ✓ Found 40 epochs
  Parsing: histgen_training_17_seed43.o1309038
  Found 40 epoch blocks
  Successfully parsed epochs: [1, 2, 3, 4, 5]... (first 5)
  Sample train_loss: [0.6467702123342822, 0.2952431857583938, 0.24887153145653398]
  Sample val_BLEU_4: [0.506255714617869, 0.5806020653046644, 0.5572015902492659]
    ✓ Found 40 epochs
  Parsing: histgen_training_17_seed444444.o1309317
  Found 40 epoch blocks
  Successfully parsed epochs: [1, 2, 3, 4, 5]... (first 5)
  Sample train_loss: [0.6488167282051029, 0.2959822161802278, 0.24719265704474208]
  Sample val_BLEU_4: [0.48873624224817064, 0.4311559541065247, 0.48188202470843206]
    ✓ Fo